# The Knowledge Analyst (RAG Document Intelligence)

In [ ]:
# Objective: Simulate a Retrieval-Augmented Generation (RAG) workflow to extract Risks, Dates, and Stakeholders from a legal document, with citation-forced AI responses
# Document Used: Google Terms of Service (Effective May 22, 2024)

In [9]:
# Installing required libraries
!pip install google-generativeai pypdf python-dotenv

In [10]:
# Load API key from .env file
from dotenv import load_dotenv
import os

load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")

if api_key:
    print("API key loaded successfully!")
else:
    print("API key not found. Check .env file")

API key loaded successfully!


In [11]:
# Load and read the PDF document
from pypdf import PdfReader

# Change this filename to analyze any PDF document
PDF_FILE = "google_terms_of_service_en.pdf"

reader = PdfReader(PDF_FILE)
total_pages = len(reader.pages)
print(f"PDF loaded successfully! Total pages: {total_pages}")

PDF loaded successfully! Total pages: 20


In [12]:
# Extract text from each page, keeping track of page numbers
pages = []

for i, page in enumerate(reader.pages):
  text = page.extract_text()
  pages.append({
    "page_number": i+1,
    "content": text
  })
print(f"Successfully extracted text from {len(pages)} pages")
print(f"\nSample from page 1:\n{pages[0]['content'][:300]}")

Successfully extracted text from 20 pages

Sample from page 1:
GOOGLE TERMS OF SERVICE
Effective May 22, 2024 | Archived versions
W hat’s covered in these term s
W e know it’s tem pting to skip these Term s of
S ervice, but it’s important to establish what you
can expect from  us as you use Google services,
and what we expect from  you.
These Terms of Service r


In [14]:
# Connect to Google's Generative AI
import google.generativeai as genai

genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-1.5-flash")
print("Gemini model loaded successfully!")

Gemini model loaded successfully!


##### Note: During development, a deprecation warning was encountered indicating that 'google.generativeai' is no longer being maintained. To ensure the project uses a supported and future-proof approach, the code was updated to use the newer 'google.genai' client-based package going forward.

In [13]:
# Switching to the newer supported google.genai package
!pip install google-genai

In [14]:
# Initialize the Gemini client using the newer google.genai package
from google import genai

client = genai.Client(api_key=api_key)
print("Gemini client initialized successfully!")

Gemini client initialized successfully!


In [15]:
# Retrieve the most relevant pages for a given query
# This simulates the 'Retrieval' part of RAG
def get_relevant_pages(query, pages, top_n=3):
    results=[]
    query_words = set(query.lower().split())

    for page in pages:
        content_words = set(page["content"].lower().split())
        overlap = len(query_words.intersection(content_words))
        results.append((overlap, page["page_number"], page["content"]))

    results.sort(reverse=True)
    return results[:top_n]
print("Retrieval function ready")

Retrieval function ready


In [22]:
# Listing available models to identify the best free tier option
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-max-preview-04-2026
models/deep-research-preview-04-2026
models/deep-resea

In [49]:
# Send query to Gemini with retrieved pages, forcing citation of page numbers
def ask_with_citation(query):
    relevant_pages = get_relevant_pages(query, pages)
    context = ""

    for page in relevant_pages:
        context += f"\n[Page {page[1]}]\n{page[2]}\n"

    prompt = f"""You are a legal document analyst. Using ONLY the context provided below,
answer the question. You MUST cite the exact page number for every claim you make. Format
citations as (Page X).

Context:
{context}

Question: {query}

Answer:"""

    response = client.models.generate_content(
        model = "gemini-flash-lite-latest",
        contents = prompt
    )
    return response.text
print("Query function ready.")

Query function ready.


In [32]:
# Test the RAG pipeline with a sample query
response = ask_with_citation("What are the age requirements for using Google services?")
print(response)

Based on the provided context, the exact age numbers are not specified; however, the following rules regarding age requirements are mentioned:

*   Users may create a Google Account if they "meet these age requirements" (Page 10).
*   "Service-specific additional terms" may include "additional age requirements" (Page 4).


## Summary Dashboard
##### Automatically extracts key information from the document (Risks, Dates, and Stakeholders) simulating how an AI-powered document intelligence tool would work in a real legal or business context

In [35]:
# Extract Risks, Dates, and Stakeholders from the entire document
def extract_summary(category):
    full_text = ""

    for page in pages:
        full_text += f"\n[Page {page['page_number']}]\n{page['content']}\n"
    
    prompt = f"""You are a legal document analyst. Analyze the document below and extract all
{category} mentioned. For each item found, cite the exact page number as (Page X). Be thorough
and specific.

Document:
{full_text}

Extract all {category}:"""

    response = client.models.generate_content(
        model = "gemini-flash-latest",
        contents = prompt
    )
    return response.text

print("Summary extraction function ready")

Summary extraction function ready


In [58]:
# Running the dashboard extraction for all three categories
print("=" * 150)
print("\t\t\t\t\t\t\t\tRISKS")
print("=" * 150)
print(extract_summary("Risks"))

print("\n" + "=" * 150)
print("\t\t\t\t\t\t\t\tDATES")
print("=" * 150)
print(extract_summary("Dates"))

print("\n" + "=" * 150)
print("\t\t\t\t\t\t\t\tSTAKEHOLDERS")
print("=" * 150)
print(extract_summary("Stakeholders"))

								RISKS
Based on the provided Google Terms of Service, the following risks are identified:

### **1. Service Changes and Discontinuation**
*   **Loss of Features or Functionality:** Google may add or remove features, increase or decrease limits to services, and stop offering old services at any time. (Page 3)
*   **Automatic Software Updates:** Downloadable or preloaded software may update automatically on your device, which may change how the service functions. (Page 3)
*   **Abrupt Service Interruption:** While Google generally provides advance notice for material changes, they may stop offering a service or make changes without notice in urgent situations such as preventing abuse, responding to legal requirements, or addressing security issues. (Page 4)

### **2. Intellectual Property and Content Licensing**
*   **Broad License Grant to Google:** By using the services, you grant Google a worldwide, non-exclusive, royalty-free license to host, reproduce, distribute, communicate

## RAG Pipeline: Citation-Forced Q&A


In [59]:
# Testing the RAG pipeline with targeted questions
questions = [
    "What happens if a user violates Google's terms?",
    "What are Google's liability limitations",
    "What rights does Google have over user content?"
]
for q in questions:
    print("\n" + "=" * 150 + "\n")
    print(f"Q: {q}" + "\n")
    print(f"A: {ask_with_citation(q)}")
    
print("\n" + "=" * 150 + "\n")



Q: What happens if a user violates Google's terms?

A: If a user violates Google’s terms, the following consequences and conditions apply:

*   **Indemnification (Business Users/Organizations):** If you are a business user or organization, you must indemnify Google and its associates against any third-party legal proceedings resulting from your violation of these terms or service-specific additional terms (Page 14). This obligation covers liabilities, expenses, claims, losses, damages, judgments, fines, litigation costs, and legal fees, unless those expenses were caused by Google's own breach, negligence, or willful misconduct (Page 14).
*   **Account Action:** Google may disable accounts for activities such as misleading others or scraping content that does not belong to you (Page 16). Users whose accounts have been suspended or terminated can appeal if they believe the action was taken in error (Page 16).
*   **Legal Exemptions:** If a user is legally exempt from responsibilities s

## Conclusion
##### This notebook successfully demonstrates a Retrieval-Augmented Generation (RAG) workflow applied to a real legal document - Google's Terms of Service (May 2024).

##### Key capabilities demonstrated:
##### - PDF loading and page-level text extraction
##### - Keyword-based retrieval to identify relevant pages
##### - Citation-forced AI responses using engineered prompts
##### - Automated Summary Dashboard extracting Risks, Dates, and Stakeholders

##### Model used: Google Gemini ('gemini-flash-lite-latest') via the 'google.genai' package  
##### Document: Google Terms of Service (20 pages)